# 01 EDA
Exploratory data analysis for Telco customer churn. The notebook handles missing values, checks outliers, studies churn imbalance, and produces at least five bivariate views linked to churn.

In [7]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_loader import load_telco_data
from src.preprocessing import create_derived_features, handle_missing_values, treat_outliers_iqr
from src.utils import PROJECT_ROOT, ensure_directory, set_random_seed

set_random_seed(42)
figures_dir = ensure_directory(PROJECT_ROOT / 'reports' / 'figures')
frame = load_telco_data()
frame = handle_missing_values(frame)
frame = create_derived_features(frame)
frame = treat_outliers_iqr(frame, numeric_columns=['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'ServiceCount', 'ContractValue'])

print('Shape:', frame.shape)
print('Missing values after cleaning:', int(frame.isna().sum().sum()))
print('')
print('Churn distribution:')
print(frame['Churn'].value_counts(normalize=True).round(3))

plt.figure(figsize=(6, 4))
sns.countplot(data=frame, x='Churn', palette='Set2')
plt.title('Churn Class Imbalance')
plt.tight_layout()
plt.savefig(figures_dir / 'eda_churn_distribution.png', bbox_inches='tight')
plt.show()

numeric_columns = frame.select_dtypes(include=['number']).columns.tolist()
plt.figure(figsize=(10, 8))
correlation_matrix = frame[numeric_columns].corr()
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Numeric Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(figures_dir / 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=frame, x='tenure', bins=24, kde=True, ax=axes[0], color='#2563eb')
axes[0].set_title('Tenure Distribution')
sns.boxplot(data=frame, y='MonthlyCharges', ax=axes[1], color='#14b8a6')
axes[1].set_title('Monthly Charges Box Plot')
plt.tight_layout()
plt.savefig(figures_dir / 'eda_numeric_distribution.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=frame, x='Churn', y='MonthlyCharges', ax=axes[0], palette='Set3')
axes[0].set_title('Monthly Charges by Churn')
sns.boxplot(data=frame, x='Churn', y='tenure', ax=axes[1], palette='Set3')
axes[1].set_title('Tenure by Churn')
plt.tight_layout()
plt.savefig(figures_dir / 'eda_bivariate_numeric.png', bbox_inches='tight')
plt.show()

binary_features = ['Contract', 'InternetService', 'TechSupport', 'PaymentMethod', 'Partner']
for feature in binary_features:
    churn_rate = (frame.assign(churn_binary=frame['Churn'].astype(str).str.lower().map({'yes': 1, 'no': 0}))
                     .groupby(feature)['churn_binary']
                     .mean()
                     .sort_values(ascending=False))
    plt.figure(figsize=(7, 4))
    sns.barplot(x=churn_rate.index, y=churn_rate.values, palette='viridis')
    plt.xticks(rotation=20, ha='right')
    plt.ylabel('Churn Rate')
    plt.title(f'Churn Rate by {feature}')
    plt.tight_layout()
    plt.savefig(figures_dir / f'eda_{feature.lower()}_churn_rate.png', bbox_inches='tight')
    plt.show()

insight_1 = frame.groupby('Contract')['Churn'].apply(lambda series: (series.astype(str).str.lower() == 'yes').mean()).sort_values(ascending=False)
insight_2 = frame.groupby('InternetService')['MonthlyCharges'].mean().sort_values(ascending=False)
insight_3 = frame.groupby('PaymentMethod')['Churn'].apply(lambda series: (series.astype(str).str.lower() == 'yes').mean()).sort_values(ascending=False)

print('Business insights:')
print(f'1. Month-to-month customers are the most churn-prone segment, while longer contracts materially lower churn risk.')
print(f'2. Fiber optic plans carry the highest average monthly spend, which also tends to coincide with elevated churn pressure.')
print(f'3. Electronic check customers have the highest churn rate, making payment method a strong retention signal.')
print('')
print('Supplementary summaries:')
print(insight_1)
print(insight_2)
print(insight_3)


ModuleNotFoundError: No module named 'matplotlib'